# Project 3: Eliminating Child Care Deserts in New York State through Optimization

In [101]:
import numpy as numpy
import pandas as pd

### Data Cleaning

In [102]:
population = pd.read_csv("./data/population.csv")
employment = pd.read_csv("./data/employment_rate.csv")
income = pd.read_csv("./data/avg_individual_income.csv")
facilities = pd.read_csv("./data/child_care_regulated.csv")
locations = pd.read_csv("./data/potential_locations.csv")

In [103]:
income.rename(columns={
    "ZIP code" : "zipcode"
}, inplace=True)

#### Defining Demographic Information
* r_a: the rate of employment of area a
* e_a: the average income level of area a
* p_t_a : the total number of children ages 2 weeks - 12 years in area a
* p_u_a : the total number of children under 5 (babies and toddlers) in area a


In [104]:
df = employment.copy()
df = df.merge(income, on="zipcode")
df["p_t_a"] = population["-5"] + population["5-9"] + (3/5)*population["10-14"]
df["p_u_a"] = population["-5"]
df.rename(columns={"employment rate":"r_a","average income":"e_a"}, inplace=True)
df

,zipcode,r_a,e_a,p_t_a,p_u_a
0,10001,0.595097,102878.033603,4.0,0
1,10002,0.520662,59604.041165,2093.2,744
2,10003,0.497244,114273.049645,7106.8,2142
3,10004,0.506661,132004.310345,3045.8,1440
4,10005,0.665833,121437.713311,711.6,433
...,...,...,...,...,...
1370,14767,0.322296,54623.287671,31.6,0
1371,14770,0.446676,55523.255814,28.0,0
1372,14772,0.410719,57164.634146,891.6,375
1373,14805,0.679739,59375.000000,141.2,20


Flagging high-demand areas (defined as regions where at least 60% of parents are employed or the average income is $60,000 or less per year)

In [105]:
df["high_demand"] = (df["r_a"] >= 0.60) | (df["e_a"] <= 60000)
df

,zipcode,r_a,e_a,p_t_a,p_u_a,high_demand
0,10001,0.595097,102878.033603,4.0,0,False
1,10002,0.520662,59604.041165,2093.2,744,True
2,10003,0.497244,114273.049645,7106.8,2142,False
3,10004,0.506661,132004.310345,3045.8,1440,False
4,10005,0.665833,121437.713311,711.6,433,True
...,...,...,...,...,...,...
1370,14767,0.322296,54623.287671,31.6,0,True
1371,14770,0.446676,55523.255814,28.0,0,True
1372,14772,0.410719,57164.634146,891.6,375,True
1373,14805,0.679739,59375.000000,141.2,20,True


#### Defining Operational Information
- n_t_a_j : the total number of existing slots in facility j in area a for all age ranges
- n_u_a_j : the number of existing slots in facility j in area a for child under 5

Assumption: Here, we assumed that infants - preschool accounts for the number of children under 5

In [106]:
facilities.rename(columns={"zip_code":"zipcode","total_capacity": "n_t_a_j"}, inplace=True)
facilities["n_u_a_j"] = (facilities["infant_capacity"] + facilities["toddler_capacity"] + facilities["preschool_capacity"])
facilities

,facility_id,program_type,facility_status,facility_name,city,zipcode,school_district_name,infant_capacity,toddler_capacity,preschool_capacity,school_age_capacity,children_capacity,n_t_a_j,latitude,longitude,n_u_a_j
0,2416,FDC,Registration,"Bohrer, Barbara",Clinton,13323,Clinton,0,0,0,2,6,8,NaN,NaN,0
1,5555,FDC,Registration,"Matey, Sally",Jamestown,14701,Jamestown,0,0,0,2,6,8,NaN,NaN,0
2,9066,FDC,Registration,"Copeland, Denise",Wappingers Falls,12590,Wappingers,0,0,0,2,6,8,NaN,NaN,0
3,40163,DCC,License,"Head Start of Rockland, Inc.",Nyack,10960,Nyack,0,10,110,0,0,120,41.089425,-73.920413,120
4,41016,SACC,Registration,"School's Out, Inc.",Glenmont,12077,Bethlehem,0,0,0,75,0,75,42.607043,-73.788606,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
15599,892735,GFDC,License,LITTLE LILIES GROUP FAMILY DAYCARE LLC.,Bronx,10462,Bronx 11,0,0,0,4,12,16,40.854317,-73.864996,0
15600,897263,GFDC,License,"Cummings, Darlene",Brooklyn,11205,Brooklyn 13,0,0,0,0,10,10,40.696940,-73.977121,0
15601,901966,GFDC,License,"Pascal Genao, Angela",Yonkers,10705,Yonkers,0,0,0,4,12,16,40.910754,-73.893528,0
15602,892455,GFDC,License,"Warnakulasuriya, Sajeeka",Staten Island,10303,Richmond 31,0,0,0,4,12,16,40.628144,-74.156228,0


Now we group the dataset by the zipcodes, so n_t_a_j becomes n_t_a, and n_u_a_j becomes n_u_a.

In [107]:
temp_facilities = facilities.groupby("zipcode").agg({"n_t_a_j": "sum","n_u_a_j": "sum"}).reset_index()
temp_facilities.rename(columns={"n_t_a_j":"n_t_a","n_u_a_j":"n_u_a"}, inplace=True)
temp_facilities

,zipcode,n_t_a,n_u_a
0,10001,609,0
1,10002,4729,18
2,10003,1995,0
3,10004,263,0
4,10005,39,0
...,...,...,...
1599,148799621,8,0
1600,149011149,16,0
1601,149012216,82,82
1602,149042203,16,0


In [108]:
df = df.merge(temp_facilities, on="zipcode", how="left")
df["n_t_a"] = df["n_t_a"].fillna(0)
df["n_u_a"] = df["n_u_a"].fillna(0)
df

,zipcode,r_a,e_a,p_t_a,p_u_a,high_demand,n_t_a,n_u_a
0,10001,0.595097,102878.033603,4.0,0,False,609.0,0.0
1,10002,0.520662,59604.041165,2093.2,744,True,4729.0,18.0
2,10003,0.497244,114273.049645,7106.8,2142,False,1995.0,0.0
3,10004,0.506661,132004.310345,3045.8,1440,False,263.0,0.0
4,10005,0.665833,121437.713311,711.6,433,True,39.0,0.0
...,...,...,...,...,...,...,...,...
1370,14767,0.322296,54623.287671,31.6,0,True,16.0,0.0
1371,14770,0.446676,55523.255814,28.0,0,True,70.0,15.0
1372,14772,0.410719,57164.634146,891.6,375,True,108.0,32.0
1373,14805,0.679739,59375.000000,141.2,20,True,8.0,0.0


#### Child Care Desert Classification
- In high-demand areas: if the number of available slots is less than or equal to half the population of children aged two weeks to 12 years
- In normal-demand areas: if the available slots are less than or equal to one-third of the population of children within the same age range

In [109]:
df["is_desert"] = df.apply(lambda row: row["n_t_a"] <= 0.5 * row["p_t_a"] if row["high_demand"] else row["n_t_a"] <= (1/3) * row["p_t_a"],axis=1)
df

,zipcode,r_a,e_a,p_t_a,p_u_a,high_demand,n_t_a,n_u_a,is_desert
0,10001,0.595097,102878.033603,4.0,0,False,609.0,0.0,False
1,10002,0.520662,59604.041165,2093.2,744,True,4729.0,18.0,False
2,10003,0.497244,114273.049645,7106.8,2142,False,1995.0,0.0,True
3,10004,0.506661,132004.310345,3045.8,1440,False,263.0,0.0,True
4,10005,0.665833,121437.713311,711.6,433,True,39.0,0.0,True
...,...,...,...,...,...,...,...,...,...
1370,14767,0.322296,54623.287671,31.6,0,True,16.0,0.0,False
1371,14770,0.446676,55523.255814,28.0,0,True,70.0,15.0,False
1372,14772,0.410719,57164.634146,891.6,375,True,108.0,32.0,True
1373,14805,0.679739,59375.000000,141.2,20,True,8.0,0.0,True


#### Age 0-5 Child Care Desert Classification
* Children under the age of 5 must have sufficient access to care
* The number of available slots for children in this age group must be at least two-thirds of the population of children aged 0-5



In [110]:
df["is_under5_desert"] = df["n_u_a"] < (2/3) * df["p_u_a"]
df

,zipcode,r_a,e_a,p_t_a,p_u_a,high_demand,n_t_a,n_u_a,is_desert,is_under5_desert
0,10001,0.595097,102878.033603,4.0,0,False,609.0,0.0,False,False
1,10002,0.520662,59604.041165,2093.2,744,True,4729.0,18.0,False,True
2,10003,0.497244,114273.049645,7106.8,2142,False,1995.0,0.0,True,True
3,10004,0.506661,132004.310345,3045.8,1440,False,263.0,0.0,True,True
4,10005,0.665833,121437.713311,711.6,433,True,39.0,0.0,True,True
...,...,...,...,...,...,...,...,...,...,...
1370,14767,0.322296,54623.287671,31.6,0,True,16.0,0.0,False,False
1371,14770,0.446676,55523.255814,28.0,0,True,70.0,15.0,False,False
1372,14772,0.410719,57164.634146,891.6,375,True,108.0,32.0,True,True
1373,14805,0.679739,59375.000000,141.2,20,True,8.0,0.0,True,True


### Export Cleaned Data to csv

In [111]:
df.to_csv("cleaned_data.csv", index=False)
df

,zipcode,r_a,e_a,p_t_a,p_u_a,high_demand,n_t_a,n_u_a,is_desert,is_under5_desert
0,10001,0.595097,102878.033603,4.0,0,False,609.0,0.0,False,False
1,10002,0.520662,59604.041165,2093.2,744,True,4729.0,18.0,False,True
2,10003,0.497244,114273.049645,7106.8,2142,False,1995.0,0.0,True,True
3,10004,0.506661,132004.310345,3045.8,1440,False,263.0,0.0,True,True
4,10005,0.665833,121437.713311,711.6,433,True,39.0,0.0,True,True
...,...,...,...,...,...,...,...,...,...,...
1370,14767,0.322296,54623.287671,31.6,0,True,16.0,0.0,False,False
1371,14770,0.446676,55523.255814,28.0,0,True,70.0,15.0,False,False
1372,14772,0.410719,57164.634146,891.6,375,True,108.0,32.0,True,True
1373,14805,0.679739,59375.000000,141.2,20,True,8.0,0.0,True,True
